# Grade-4 auxiliary loss — controlled training experiment

**Hypothesis.** The CORN objective does not put sufficient training pressure on the Grade-3/Grade-4
boundary, so useful lesion information already present in the inputs is discarded from the learned
`p_gt_3` score.

**Comparison.** `corn_loss` (baseline) vs `corn_loss + lambda * BCE(p_gt_3, grade == 4)` (auxiliary),
with lambda pre-registered as 0.05 / 0.10 / 0.20 and seeds 42 / 123 / 2026. Nothing else changes.

**How to use.** Run all cells. The notebook finds the first unfinished run in the fixed 12-run
manifest, resumes or starts it, evaluates it, marks it complete and moves on to the next. After a
disconnect, run all cells again — it continues where it stopped. **No seed or lambda is ever edited
by hand.**

*The threshold pathway remains a separate calibration finding. This experiment tests whether
changing the training objective improves the underlying Grade-3/Grade-4 representation.*

In [ ]:
# ==== [1] ENVIRONMENT / IMPORTS ====
# The project's standard bootstrap: clone/pull the repository, then mount Drive, install
# requirements and verify the environment through colab/common/setup.py. A GPU is required:
# every auxiliary run trains the full joint model.
import os
import posixpath
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

import setup as colab_setup
setup_info = colab_setup.setup()

import colab_config
import verify_environment
env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR, drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"), require_gpu=True)

import csv
import datetime
import gc
import json
import shutil
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import roc_auc_score

import config
import corn
import improved_training_data as itd
import multiseed_runs as msr
import weighted_corn
from training import (Trainer, TrainingConfig, CheckpointOptions, TrainingStateCheckpoint,
                      expected_policy_name, model_precision_policies)
from training import checkpointing as ckpt
from training.trainer import precision_is_consistent

print("Ready. multiseed_runs protocol:", msr.PROTOCOL_VERSION)

In [ ]:
# ==== [2] CONFIGURATION AND FROZEN PRE-REGISTRATION ====
# Everything in PREREGISTRATION is frozen the first time this cell runs. On every later run the
# stored file is re-verified and compared: if any value here has been edited, the notebook refuses
# to continue. That is what stops lambda or a threshold being tuned after results are seen.

EXPERIMENT_ID = "grade4_aux_loss_2026_09"
LAMBDAS = (0.05, 0.10, 0.20)      # PRE-REGISTERED. Do not add, remove or change.
SEEDS = (42, 123, 2026)           # PRE-REGISTERED. The six-run experiment's matched seeds.
AUX_LOSS_VERSION = "grade4-bce-on-marginal-p_gt_3-unweighted-v1"   # bump if [6] ever changes
BCE_EPSILON = 1e-6
PDR_GRADE, SEVERE_NPDR_GRADE = 4, 3

# Decision rule, fixed before any auxiliary run is trained.
#  * Primary outcome: AUROC of p_gt_3 on Grade 4 vs Grade 3 -- NOT QWK, NOT Grade-4 recall, NOT a
#    tuned threshold.
#  * MIN_MEAN_AUROC_GAIN = 0.03 is ~40% of the 0.072 gap between p_gt_3 (0.598) and the lesion-only
#    probe (0.670) recorded in the research record, section 11.
#  * All three seeds must move in the same direction: with 39 vs 58 cases the Hanley-McNeil SE of a
#    single AUROC is ~0.058, so one seed alone is noise-level. Mirrors the six-run experiment's own
#    sign-consistency rule.
MIN_MEAN_AUROC_GAIN = 0.03
REQUIRED_POSITIVE_SEEDS = 3
MAX_MEAN_QWK_DROP = 0.02
MAX_MEAN_MAE_RISE = 0.03
MIN_LAMBDAS_FOR_OVERALL = 2       # guards against one lucky lambda out of three
GRADE3_RECALL_CAUTION = -0.10     # reported, not decisive: research record section 10 lesson

EXPERIMENTS_ROOT = colab_config.DRIVE.experiments_root
SIX_RUN_ROOT = posixpath.join(EXPERIMENTS_ROOT, "ImprovedTraining")
SIX_RUN_ID = "improved_multiseed_2026_09"
SIX_RUN_DIR = msr.experiment_root(SIX_RUN_ROOT, SIX_RUN_ID)
EXPERIMENT_DIR = posixpath.join(EXPERIMENTS_ROOT, "Grade4AuxLoss", EXPERIMENT_ID)

for _protected in (SIX_RUN_DIR,
                   posixpath.join(EXPERIMENTS_ROOT, "ImprovedTrainingC1Control"),
                   posixpath.join(EXPERIMENTS_ROOT, "ImprovedTrainingC3Kappa"),
                   colab_config.DRIVE.experiment_dir("FinalClassification")):
    assert posixpath.commonpath([posixpath.normpath(EXPERIMENT_DIR),
                                 posixpath.normpath(_protected)]) != posixpath.normpath(_protected), (
        f"{EXPERIMENT_DIR} resolves under protected path {_protected} -- refusing.")
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# Training protocol copied from the six-run experiment's own frozen pre-registration, so every run
# here uses exactly the protocol the baselines were trained under.
SIX_RUN_PREREG, _ = msr.load_and_verify_preregistration(
    posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
BATCH_SIZE = SIX_RUN_PREREG["batch_size"]
MAX_EPOCHS = SIX_RUN_PREREG["max_epochs"]
LEARNING_RATE = SIX_RUN_PREREG["learning_rate"]
WEIGHT_DECAY = SIX_RUN_PREREG["weight_decay"]
CLASS_WEIGHTS = list(SIX_RUN_PREREG["class_weights"])
MONITOR, MODE = SIX_RUN_PREREG["primary_metric"], "max"
EARLY_STOPPING_PATIENCE = SIX_RUN_PREREG["early_stopping"]["patience"]
REDUCE_LR_PATIENCE = SIX_RUN_PREREG["reduce_lr_on_plateau"]["patience"]
REDUCE_LR_FACTOR = SIX_RUN_PREREG["reduce_lr_on_plateau"]["factor"]
MIN_LR = SIX_RUN_PREREG["reduce_lr_on_plateau"]["min_lr"]

with open(posixpath.join(SIX_RUN_DIR, "experiment_manifest.json")) as _fh:
    SIX_RUN_POPULATION = json.load(_fh)
EXPECTED_VAL_N = int(SIX_RUN_POPULATION["n_val_yielded"])

PREREGISTRATION = {
    "experiment_id": EXPERIMENT_ID,
    "hypothesis": ("The CORN objective does not provide sufficient training pressure on the "
                   "Grade-3/Grade-4 boundary, causing useful lesion information already present in "
                   "the inputs to be discarded from the learned p_gt_3 score."),
    "lambdas": list(LAMBDAS), "seeds": list(SEEDS),
    "loss_baseline": "weighted_corn_loss (unchanged)",
    "loss_auxiliary": ("weighted_corn_loss + lambda * BCE(p_gt_3, 1[grade == 4]); p_gt_3 is the "
                       "marginal P(y = 4) = prod_j sigmoid(logit_j); BCE unweighted; epsilon "
                       f"{BCE_EPSILON}"),
    "aux_loss_version": AUX_LOSS_VERSION,
    "baseline_policy": ("frozen NO_RACAF runs of improved_multiseed_2026_09 at the same seeds are "
                        "the lambda = 0 arm and are reused, not retrained, whenever their artifacts "
                        "are valid; otherwise the baseline is retrained with the identical "
                        "msr.train_run path"),
    "model": "no_racaf_model.build_no_racaf_joint_model_matched_init via msr.build_arm_model",
    "checkpoint_selection": f"{MONITOR} ({MODE}), unchanged",
    "protocol_from_six_run": {
        "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "class_weights": CLASS_WEIGHTS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "reduce_lr_patience": REDUCE_LR_PATIENCE, "reduce_lr_factor": REDUCE_LR_FACTOR,
        "min_lr": MIN_LR, "split_sha256": msr.EXPECTED_SPLIT_SHA256},
    "primary_outcome": "AUROC of p_gt_3 on Grade 4 vs Grade 3 (not QWK, not recall, not thresholds)",
    "decision_rule": {
        "boundary_improved": (f"mean delta AUROC >= {MIN_MEAN_AUROC_GAIN} AND delta > 0 in "
                              f"{REQUIRED_POSITIVE_SEEDS}/3 seeds"),
        "performance_preserved": (f"mean delta QWK >= -{MAX_MEAN_QWK_DROP} AND mean delta MAE <= "
                                  f"+{MAX_MEAN_MAE_RISE}"),
        "per_lambda": ("SUPPORTIVE = improved and preserved; TRADE_OFF = improved, not preserved; "
                       "NOT_SUPPORTIVE = not improved"),
        "overall": (f"SUPPORTIVE if >= {MIN_LAMBDAS_FOR_OVERALL} lambdas SUPPORTIVE; TRADE_OFF if "
                    f">= {MIN_LAMBDAS_FOR_OVERALL} lambdas improve the boundary but fewer are "
                    "SUPPORTIVE; otherwise NOT_SUPPORTIVE. No lambda is selected retrospectively."),
        "grade3_recall_caution": GRADE3_RECALL_CAUTION},
    "manifest_order": ["baseline seed42", "baseline seed123", "baseline seed2026"]
                      + [f"lambda{lam:.2f} seed{seed}" for lam in LAMBDAS for seed in SEEDS],
}

PREREG_PATH = posixpath.join(EXPERIMENT_DIR, "PREREGISTRATION.json")
_expected = json.loads(json.dumps(PREREGISTRATION))
if os.path.exists(PREREG_PATH):
    _stored, _digest = msr.load_and_verify_preregistration(PREREG_PATH)
    if {k: v for k, v in _stored.items() if k != "frozen_on"} != _expected:
        raise RuntimeError(
            f"{PREREG_PATH} differs from this notebook's pre-registration. The pre-registration is "
            "frozen; revert the edit rather than changing a lambda, seed or threshold mid-experiment.")
    print(f"Pre-registration verified (frozen on {_stored.get('frozen_on')}).")
else:
    _payload = dict(_expected, frozen_on=datetime.datetime.now().isoformat(timespec="seconds"))
    with open(PREREG_PATH, "w") as _fh:
        _fh.write(json.dumps(_payload, indent=2, sort_keys=True))
    print(f"Pre-registration FROZEN now at {PREREG_PATH}.")

In [ ]:
# ==== [3] EXPERIMENT MANIFEST ====
# The fixed run order. Each run has its own stable directory naming its arm, lambda and seed.

def lambda_tag(lam):
    return f"{lam:.2f}"


MANIFEST = []
for _seed in SEEDS:
    MANIFEST.append({"kind": "baseline", "lambda": 0.0, "seed": _seed,
                     "label": f"baseline seed{_seed}",
                     "dir": posixpath.join(EXPERIMENT_DIR, "baseline", f"seed_{_seed}")})
for _lam in LAMBDAS:
    for _seed in SEEDS:
        MANIFEST.append({"kind": "aux", "lambda": _lam, "seed": _seed,
                         "label": f"lambda{lambda_tag(_lam)} seed{_seed}",
                         "dir": posixpath.join(EXPERIMENT_DIR, f"aux_lambda_{lambda_tag(_lam)}",
                                               f"seed_{_seed}")})
for _index, _run in enumerate(MANIFEST, start=1):
    _run["index"] = _index
assert [r["label"] for r in MANIFEST] == PREREGISTRATION["manifest_order"]

for _run in MANIFEST:
    print(f"{_run['index']:>2}. {_run['label']:<22} {_run['dir']}")

In [ ]:
# ==== [4] RESUME / RUN DISCOVERY ====
# Status is derived from what is on disk every time -- never from a counter that could drift.
#   COMPLETED_FROZEN_REFERENCE  baseline served by the frozen six-run NO_RACAF artifacts
#   COMPLETED                   trained here, stopped, evaluated
#   NEEDS_EVALUATION            stopped, evaluation not yet written (crash between the two)
#   INTERRUPTED                 checkpoints exist, not stopped -> resumes from the last valid epoch
#   NOT_STARTED

def frozen_baseline_dir(seed):
    return msr.run_dir(SIX_RUN_ROOT, SIX_RUN_ID, "NO_RACAF", seed)


def per_sample_valid(path):
    if not os.path.exists(path):
        return False
    frame = pd.read_csv(path, usecols=["image_id", "true_grade", "predicted_grade", "p_gt_3"])
    return len(frame) == EXPECTED_VAL_N


def run_status(run):
    if run["kind"] == "baseline":
        frozen = frozen_baseline_dir(run["seed"])
        if (msr.read_stop_decision(frozen) is not None and per_sample_valid(
                posixpath.join(frozen, "evaluation", "per_sample_best.csv"))):
            return "COMPLETED_FROZEN_REFERENCE"
    run_dir = run["dir"]
    if msr.read_stop_decision(run_dir) is not None:
        if (os.path.exists(posixpath.join(run_dir, "evaluation", "metrics_best.json"))
                and per_sample_valid(posixpath.join(run_dir, "evaluation", "per_sample_best.csv"))):
            return "COMPLETED"
        return "NEEDS_EVALUATION"
    if ckpt.checkpoint_evidence(posixpath.join(run_dir, "checkpoints")) or msr.read_history(run_dir):
        return "INTERRUPTED"
    return "NOT_STARTED"


def result_dir(run):
    """Where a completed run's artifacts live: the frozen six-run directory for a reused baseline,
    otherwise the run's own directory in this experiment."""
    if run_status(run) == "COMPLETED_FROZEN_REFERENCE":
        return frozen_baseline_dir(run["seed"])
    return run["dir"]


def write_status_table():
    table = []
    for run in MANIFEST:
        status = run_status(run)
        table.append({"index": run["index"], "label": run["label"], "status": status,
                      "result_dir": result_dir(run) if status.startswith("COMPLETED") else run["dir"]})
        if run["kind"] == "baseline" and status == "COMPLETED_FROZEN_REFERENCE":
            os.makedirs(run["dir"], exist_ok=True)
            with open(posixpath.join(run["dir"], "reference.json"), "w") as fh:
                json.dump({"reused_frozen_baseline": frozen_baseline_dir(run["seed"]),
                           "reason": PREREGISTRATION["baseline_policy"]}, fh, indent=2)
    with open(posixpath.join(EXPERIMENT_DIR, "run_status.json"), "w") as fh:
        json.dump({"updated": datetime.datetime.now().isoformat(timespec="seconds"),
                   "runs": table}, fh, indent=2)
    return table


for _row in write_status_table():
    print(f"{_row['index']:>2}. {_row['label']:<22} {_row['status']}")

In [ ]:
# ==== [5] DATA AND MODEL SETUP ====
# The authoritative split and the six-run experiment's pinned validation population are reused
# unchanged. Neither lambda nor seed can alter them.
TRAIN_ENTRIES, VAL_ENTRIES, SPLIT_SHA256 = msr.verify_split()
assert SPLIT_SHA256 == msr.EXPECTED_SPLIT_SHA256

LOCAL_CACHE_DIR = "/content/cache/local_feature_extraction"
LOCAL_RACAF_CACHE_DIR = "/content/cache/racaf"
LOCAL_CACHE_MARKER = "/content/cache/.multiseed_archive_extracted.json"
CACHE_ARCHIVE_DIR = os.path.join(os.path.dirname(config.LOCAL_FEATURE_RESULTS_DIR), "cache_archive")
DATA = {"ready": False, "train": None, "val": None}


def ensure_local_cache():
    """Identical to the six-run notebook's cache cell. Runs at most once per runtime, and only when
    some run actually needs to train or be evaluated."""
    if DATA["ready"]:
        return
    if not os.path.exists(LOCAL_CACHE_MARKER):
        import joint_cache_archive as jca
        plan = jca.plan_extraction(archive_dir=CACHE_ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                   racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
        jca.print_extraction_plan(plan)
        if plan["drive_unreachable"] or not plan["fits"]:
            raise RuntimeError("Refusing to extract the cache archive.")
        extract = jca.extract_archive(archive_dir=CACHE_ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                      racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
                                      min_free_bytes=plan["required_bytes"])
        jca.print_extract(extract)
        if extract["corrupt"] or extract["drive_unreachable"]:
            raise RuntimeError("Archive extraction did not complete. Re-run the notebook.")
        with open(LOCAL_CACHE_MARKER, "w") as fh:
            json.dump({"extracted": datetime.datetime.now().isoformat(timespec="seconds")}, fh)
    itd.complete_local_cache(
        TRAIN_ENTRIES + VAL_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR,
        config.LOCAL_FEATURE_RESULTS_DIR, config.RACAF_RESULTS_DIR,
        source_image_dir=os.path.join(colab_config.APTOS2019_RAW_DIR, "train_images"),
        local_image_dir="/content/cache/raw_images_for_cache_generation",
        known_empty_fov_ids=SIX_RUN_POPULATION.get("empty_fov_ids"),
        processed_dir="/content/cache/_no_precomputed_stage02_output")
    DATA["train"] = itd.locally_cached_entries(TRAIN_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    DATA["val"] = itd.locally_cached_entries(VAL_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    assert len(DATA["train"]) == SIX_RUN_POPULATION["n_train_yielded"], "training population drifted"
    assert len(DATA["val"]) == SIX_RUN_POPULATION["n_val_yielded"], "validation population drifted"
    assert shutil.disk_usage("/content").free >= 3 * 1024 ** 3, "less than 3 GiB free on /content"
    DATA["ready"] = True
    print(f"Population matches the six-run pin: {len(DATA['train'])} train / {len(DATA['val'])} val")


def fresh_session():
    """Called immediately before every model build. Verified locally: after a throwaway build and
    clear_session(), msr.build_arm_model reproduces a fresh-process build's initial weights
    bit-for-bit, so looping through runs in one runtime preserves matched initialisation."""
    gc.collect()
    tf.keras.backend.clear_session()


def build_model(seed, lam):
    """msr.build_arm_model('NO_RACAF', seed) -- the exact construction the frozen baselines used:
    set_random_seed, mixed precision, matched-init NO_RACAF model, AdamW, weighted CORN, and its
    structural verification. For lambda > 0 the model is then recompiled with a fresh, identical
    optimizer and the auxiliary loss; nothing else changes."""
    model = msr.build_arm_model("NO_RACAF", seed, class_weights=CLASS_WEIGHTS,
                                learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, verbose=0)
    if lam > 0:
        model.compile(
            optimizer=msr.build_optimizer(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
            loss=make_corn_plus_grade4_loss(CLASS_WEIGHTS, lam),
            metrics=[corn.CORNQuadraticWeightedKappa(), weighted_corn.UnweightedCORNLoss(),
                     Grade4AuxBCE()])
        expected = expected_policy_name(True)
        assert precision_is_consistent(expected, model_precision_policies(model))
        if expected == "mixed_float16":
            assert getattr(model.optimizer, "inner_optimizer", None) is not None, (
                "recompiled optimizer lost its LossScaleOptimizer wrapper")
    return model

In [ ]:
# ==== [6] AUXILIARY GRADE-4 LOSS ====
# The ONLY training change. CORN is neither replaced nor removed -- the auxiliary term is added.
#
#   total = weighted_corn_loss + lambda * BCE(p_gt_3, 1[grade == 4])
#
# p_gt_3 is the MARGINAL P(y = 4) = prod_j sigmoid(logit_j): the score every prior diagnostic
# measured. It is not logit_3, which is the CONDITIONAL P(y > 3 | y > 2). It is computed as a sum
# of log-sigmoids then exponentiated and clipped, so a saturated product can never produce log(0).
# The BCE is UNWEIGHTED by design: lambda is the only knob. (Grade 4 is ~8% of training, so a
# weighted auxiliary would push harder; a null result is partly attributable to this choice.)

def _p_gt_3(y_pred):
    logits = tf.cast(y_pred, tf.float32)
    log_p = tf.reduce_sum(tf.math.log_sigmoid(logits), axis=-1)
    return tf.clip_by_value(tf.exp(log_p), BCE_EPSILON, 1.0 - BCE_EPSILON)


def _grade4_bce_per_sample(y_true, y_pred):
    p = _p_gt_3(y_pred)
    target = tf.cast(tf.equal(tf.reshape(tf.cast(y_true, tf.int32), [-1]), PDR_GRADE), tf.float32)
    return -(target * tf.math.log(p) + (1.0 - target) * tf.math.log(1.0 - p))


def make_corn_plus_grade4_loss(class_weights, lam):
    base = weighted_corn.make_weighted_corn_loss(class_weights)
    lam = float(lam)

    def loss(y_true, y_pred):
        return base(y_true, y_pred) + lam * tf.reduce_mean(_grade4_bce_per_sample(y_true, y_pred))

    return loss


class Grade4AuxBCE(tf.keras.metrics.Metric):
    """Reports the auxiliary BCE (without lambda) as `grade4_aux_bce` / `val_grade4_aux_bce`. A
    metric, never a loss: it contributes no gradient. Pooled over the whole epoch, like
    weighted_corn.UnweightedCORNLoss."""

    def __init__(self, name="grade4_aux_bce", **kwargs):
        super().__init__(name=name, **kwargs)
        self.total = self.add_weight(name="total", initializer="zeros", dtype=tf.float32)
        self.count = self.add_weight(name="count", initializer="zeros", dtype=tf.float32)

    def update_state(self, y_true, y_pred, sample_weight=None):
        values = _grade4_bce_per_sample(y_true, y_pred)
        self.total.assign_add(tf.reduce_sum(values))
        self.count.assign_add(tf.cast(tf.size(values), tf.float32))

    def result(self):
        return self.total / tf.maximum(self.count, 1.0)

    def reset_state(self):
        self.total.assign(0.0)
        self.count.assign(0.0)


print("Auxiliary loss defined:", AUX_LOSS_VERSION)

In [ ]:
# ==== [7] EVALUATION ====
# Every metric is computed from the per-image BEST predictions, with the same code for reused
# baselines and new runs, so the comparison is like-for-like. The primary outcome is threshold-free.

def per_sample_metrics(frame):
    summary = msr.summarize_predictions(frame.to_dict("records"))
    true_grade = frame["true_grade"].to_numpy(dtype=int)
    score = frame["p_gt_3"].to_numpy(dtype=np.float64)
    is4 = (true_grade == PDR_GRADE).astype(int)
    mask34 = np.isin(true_grade, [SEVERE_NPDR_GRADE, PDR_GRADE])
    clipped = np.clip(score, BCE_EPSILON, 1.0 - BCE_EPSILON)
    return {
        "auroc_g4_vs_g3": float(roc_auc_score(is4[mask34], score[mask34])),
        "auroc_g4_vs_rest": float(roc_auc_score(is4, score)),
        "grade4_recall": summary["recall"][PDR_GRADE],
        "grade4_precision": summary["precision"][PDR_GRADE],
        "grade4_f1": summary["f1"][PDR_GRADE],
        "grade3_recall": summary["recall"][SEVERE_NPDR_GRADE],
        "grade3_precision": summary["precision"][SEVERE_NPDR_GRADE],
        "qwk": summary["qwk"], "accuracy": summary["accuracy"],
        "balanced_accuracy": summary["balanced_accuracy"], "macro_f1": summary["macro_f1"],
        "mae": summary["mae"], "confusion_matrix": summary["confusion_matrix"],
        "val_grade4_bce_at_best": float(np.mean(-(is4 * np.log(clipped)
                                                  + (1 - is4) * np.log(1 - clipped)))),
        "n": int(len(frame)), "n_grade3": int((true_grade == SEVERE_NPDR_GRADE).sum()),
        "n_grade4": int(is4.sum()),
    }


def history_summary(run_dir):
    history = msr.read_history(run_dir)
    pointer = msr._read_json(posixpath.join(run_dir, "checkpoints", msr.BEST_POINTER_FILENAME)) or {}
    best_index = pointer.get("epoch")
    best_row = next((h for h in history if best_index is not None
                     and h["epoch"] == best_index + 1), {})
    last_row = history[-1] if history else {}
    full_dir = posixpath.join(run_dir, "history_full")
    aux_at_best = None
    if best_row and os.path.isdir(full_dir):
        full_path = posixpath.join(full_dir, f"epoch_{best_row['epoch']:04d}.json")
        if os.path.exists(full_path):
            aux_at_best = (msr._read_json(full_path) or {}).get("val_grade4_aux_bce")
    return {
        "best_epoch_index_0based": best_index,
        "epochs_trained": last_row.get("epoch"),
        "best_val_qwk": pointer.get("val_QWK"),
        "val_loss_at_best": best_row.get("val_loss"),
        "val_corn_loss_unweighted_at_best": best_row.get("val_corn_loss_unweighted"),
        "loss_at_best": best_row.get("loss"),
        "val_grade4_aux_bce_logged_at_best": aux_at_best,
        "val_qwk_history": [h.get("val_QWK") for h in history],
        "val_loss_history": [h.get("val_loss") for h in history],
        "loss_history": [h.get("loss") for h in history],
    }


def evaluate_new_run(run):
    """BEST weights loaded into a freshly built model, evaluated on the pinned validation set, and
    written to the run's own evaluation/ directory. Nothing in any other experiment is touched."""
    ensure_local_cache()
    slot_dir, pointer = msr.read_best(run["dir"])
    assert slot_dir is not None, f"{run['label']}: no BEST published"
    fresh_session()
    model = build_model(run["seed"], run["lambda"])
    ckpt.load_model_weights_only(model, os.path.join(slot_dir, ckpt.MODEL_WEIGHTS_FILENAME))
    rows = msr.evaluate_arm_from_disk(model, DATA["val"], cache_dir=LOCAL_CACHE_DIR,
                                      racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
    eval_dir = posixpath.join(run["dir"], "evaluation")
    os.makedirs(eval_dir, exist_ok=True)
    with open(posixpath.join(eval_dir, "per_sample_best.csv"), "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    metrics = per_sample_metrics(pd.DataFrame(rows))
    metrics.update(label=run["label"], kind=run["kind"], seed=run["seed"], aux_lambda=run["lambda"],
                   best_epoch_index_0based=pointer["epoch"])
    with open(posixpath.join(eval_dir, "metrics_best.json"), "w") as fh:
        json.dump(metrics, fh, indent=2)
    del model
    print(f"  evaluated {run['label']}: g4-vs-g3 AUROC={metrics['auroc_g4_vs_g3']:.4f} "
          f"QWK={metrics['qwk']:.4f} MAE={metrics['mae']:.4f}")


def collect_result(run):
    """Metrics + history for one COMPLETED run, whether reused-frozen or trained here."""
    source = result_dir(run)
    frame = pd.read_csv(posixpath.join(source, "evaluation", "per_sample_best.csv"))
    result = per_sample_metrics(frame)
    result.update(history_summary(source))
    result.update(label=run["label"], kind=run["kind"], seed=run["seed"], aux_lambda=run["lambda"],
                  index=run["index"], status=run_status(run), result_dir=source)
    return result

In [ ]:
# ==== [8] AUTOMATIC RUN EXECUTION ====
# Walks the manifest in order. COMPLETED runs are never retrained. INTERRUPTED runs resume from the
# latest valid epoch through the project's own checkpoint machinery -- weights, optimizer (including
# the LossScaleOptimizer's dynamic scale), epoch, LR, early-stopping and LR-plateau counters, BEST
# and history. After each run: evaluate, mark complete, continue.

def acquire_lock_waiting(run_dir):
    """msr.acquire_lock never forces. If a previous runtime's lock is still fresh (a quick restart
    after a disconnect), wait for its heartbeat to go stale rather than crash -- a live runtime keeps
    refreshing it, so two runtimes can never train the same run."""
    while True:
        try:
            msr.acquire_lock(run_dir, owner_id=msr.OWNER_ID)
            return
        except msr.RunLockedError as error:
            print(f"  lock held by another runtime; waiting for it to go stale. ({error})")
            time.sleep(60)


def aux_config_mapping(seed, lam):
    return {
        "protocol_version": SIX_RUN_PREREG["protocol_version"] + "-grade4-aux",
        "experiment_id": EXPERIMENT_ID, "arm": "NO_RACAF_PLUS_GRADE4_AUX", "run_seed": seed,
        "split_seed": msr.SPLIT_SEED, "split_sha256": SPLIT_SHA256,
        "model": "joint_stage05_08_no_racaf", "aux_lambda": float(lam),
        "aux_loss_version": AUX_LOSS_VERSION, "aux_bce_epsilon": BCE_EPSILON,
        "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "weight_decay_exclude": list(msr.WEIGHT_DECAY_EXCLUDE_NAMES),
        "class_weights": list(CLASS_WEIGHTS), "monitor": MONITOR, "mode": MODE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "reduce_lr_patience": REDUCE_LR_PATIENCE, "reduce_lr_factor": REDUCE_LR_FACTOR,
        "min_lr": MIN_LR,
    }


def train_aux_run(run):
    """msr.train_run's own sequence, inlined only because train_run is gated to the six-run arms and
    its fingerprint could not see lambda. Same Trainer, checkpoints, two-slot BEST, stop decision,
    lock and training-behaviour reconciliation."""
    ensure_local_cache()
    run_dir, seed, lam = run["dir"], run["seed"], run["lambda"]
    msr.ensure_run_dir(run_dir)
    mapping = aux_config_mapping(seed, lam)
    config_hash = ckpt.config_hash(mapping)
    manifest_path = posixpath.join(run_dir, msr.RUN_MANIFEST_FILENAME)
    if os.path.exists(manifest_path):
        msr.verify_run_manifest(run_dir, mapping, config_hash)
    else:
        msr.write_run_manifest(manifest_path, mapping, config_hash, repo_dir=colab_config.REPO_DIR)

    sealed = msr._sealed_stop(posixpath.join(run_dir, "checkpoints"), MAX_EPOCHS)
    if sealed is not None:
        msr.write_stop_decision(run_dir, sealed[0], sealed[1])
        return

    fresh_session()
    model = build_model(seed, lam)
    acquire_lock_waiting(run_dir)
    try:
        sources = {}
        for relative_path, symbols in msr.TRAINING_BEHAVIOR_SOURCES:
            sources.update(msr.normalized_source_digests(relative_path, symbols, msr.REPO_ROOT))
        behaviour_config = {k: v for k, v in mapping.items()
                            if k not in ("experiment_id", "protocol_version")}
        behaviour_config["mixed_precision"] = True
        components = {"fingerprint_version": msr.TRAINING_BEHAVIOR_FINGERPRINT_VERSION,
                      "configuration": behaviour_config,
                      "train_population": msr.population_digest(DATA["train"]),
                      "validation_population": msr.population_digest(DATA["val"]),
                      "sources": sources}
        msr.reconcile_training_behavior(
            run_dir, {"training_behavior_hash": msr._canonical_hash(components),
                      "components": components},
            msr._current_git_commit(colab_config.REPO_DIR), verbose=1)

        trainer = Trainer(TrainingConfig(
            run_dir=run_dir, epochs=MAX_EPOCHS, monitor=MONITOR, mode=MODE, mixed_precision=True,
            resume=True, early_stopping_patience=EARLY_STOPPING_PATIENCE,
            reduce_lr_patience=REDUCE_LR_PATIENCE, reduce_lr_factor=REDUCE_LR_FACTOR,
            min_lr=MIN_LR, precision_check="error", repo_dir=colab_config.REPO_DIR,
            checkpoint_options=CheckpointOptions(
                experiment_id=f"aux{lambda_tag(lam)}/seed_{seed}", config_hash=config_hash,
                dataset_version="aptos2019-joint-cache-v1",
                staging_dir="/content/checkpoint_staging_grade4aux", keep_generations=2,
                verbose=1)))
        trainer.prepare(model)
        initial_epoch = trainer.resolve_initial_epoch()
        if initial_epoch > 0:
            trainer.restore(model)
            print(f"  resumed {run['label']} at epoch {initial_epoch}")
        if initial_epoch >= MAX_EPOCHS:
            msr.write_stop_decision(run_dir, initial_epoch, "epoch_cap")
            return

        early = next(c for c in trainer.callbacks
                     if isinstance(c, tf.keras.callbacks.EarlyStopping))
        state_callback = next(c for c in trainer.callbacks
                              if isinstance(c, TrainingStateCheckpoint))
        val_ds = itd.make_epoch_dataset(DATA["val"], epoch=0, run_seed=seed,
                                        cache_dir=LOCAL_CACHE_DIR,
                                        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
                                        batch_size=BATCH_SIZE, augment=False)
        full_history_dir = posixpath.join(run_dir, "history_full")
        os.makedirs(full_history_dir, exist_ok=True)
        for epoch in range(initial_epoch, MAX_EPOCHS):
            msr.heartbeat_lock(run_dir, owner_id=msr.OWNER_ID)
            train_ds = itd.make_epoch_dataset(DATA["train"], epoch=epoch, run_seed=seed,
                                              cache_dir=LOCAL_CACHE_DIR,
                                              racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
                                              batch_size=BATCH_SIZE, augment=True)
            model.fit(train_ds, validation_data=val_ds, epochs=epoch + 1, initial_epoch=epoch,
                      callbacks=trainer.callbacks, verbose=1)
            generation = state_callback.last_generation_dir
            assert generation is not None, f"epoch {epoch}: no checkpoint generation written"
            state = ckpt.read_state(generation)
            msr.write_epoch_history(run_dir, state)
            with open(posixpath.join(full_history_dir, f"epoch_{state.completed_epoch:04d}.json"),
                      "w") as fh:
                json.dump(dict((state.extra or {}).get("epoch_logs") or {},
                               epoch=state.completed_epoch, learning_rate=state.learning_rate),
                          fh, indent=2)
            if state.best_epoch == epoch:
                msr.publish_best(run_dir, generation, state, repo_dir=colab_config.REPO_DIR,
                                 verbose=1)
            if early.stopped_epoch:
                msr.write_stop_decision(run_dir, state.completed_epoch, "early_stopping")
                break
            if state.completed_epoch >= MAX_EPOCHS:
                msr.write_stop_decision(run_dir, state.completed_epoch, "epoch_cap")
                break
    finally:
        msr.release_lock(run_dir, owner_id=msr.OWNER_ID)
        del model


def train_baseline_fallback(run):
    """Only used if a frozen six-run NO_RACAF artifact is missing or invalid. msr.train_run is the
    exact function the frozen baselines were trained with."""
    ensure_local_cache()
    msr.ensure_run_dir(run["dir"])
    mapping = msr.run_config_mapping(EXPERIMENT_ID, "NO_RACAF", run["seed"], SPLIT_SHA256)
    config_hash = ckpt.config_hash(mapping)
    manifest_path = posixpath.join(run["dir"], msr.RUN_MANIFEST_FILENAME)
    if os.path.exists(manifest_path):
        msr.verify_run_manifest(run["dir"], mapping, config_hash)
    else:
        msr.write_run_manifest(manifest_path, mapping, config_hash, repo_dir=colab_config.REPO_DIR)
    fresh_session()
    model = build_model(run["seed"], 0.0)
    msr.train_run(model, run["dir"], "NO_RACAF", run["seed"], DATA["train"], DATA["val"],
                  cache_dir=LOCAL_CACHE_DIR, racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
                  config_hash_value=config_hash, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  reduce_lr_patience=REDUCE_LR_PATIENCE, reduce_lr_factor=REDUCE_LR_FACTOR,
                  min_lr=MIN_LR, staging_dir="/content/checkpoint_staging_grade4aux",
                  repo_dir=colab_config.REPO_DIR, verbose=1)
    del model


for run in MANIFEST:
    status = run_status(run)
    if status.startswith("COMPLETED"):
        continue
    print(f"\n=== [{run['index']}/{len(MANIFEST)}] {run['label']} ({status}) ===")
    if status in ("NOT_STARTED", "INTERRUPTED"):
        if run["kind"] == "baseline":
            print("  frozen baseline unavailable -> retraining it with msr.train_run")
            train_baseline_fallback(run)
        else:
            train_aux_run(run)
    if msr.read_stop_decision(run["dir"]) is not None:
        evaluate_new_run(run)
    write_status_table()
    print(f"  {run['label']}: {run_status(run)}")

print("\nRun status after this session:")
for _row in write_status_table():
    print(f"{_row['index']:>2}. {_row['label']:<22} {_row['status']}")

In [ ]:
# ==== [9] FINAL CROSS-RUN COMPARISON ====
# Paired by seed: each auxiliary run is compared with the baseline at the same seed. Every lambda is
# reported; none is selected. The verdict is labelled PARTIAL until all 12 runs are complete.

RESULTS = {run["label"]: collect_result(run) for run in MANIFEST
           if run_status(run).startswith("COMPLETED")}
N_COMPLETE = len(RESULTS)
FINAL = N_COMPLETE == len(MANIFEST)
METRICS = ["auroc_g4_vs_g3", "auroc_g4_vs_rest", "grade4_recall", "grade4_precision", "grade4_f1",
           "grade3_recall", "qwk", "accuracy", "balanced_accuracy", "macro_f1", "mae",
           "val_grade4_bce_at_best"]


def mean_sd(values):
    values = [v for v in values if v is not None]
    if not values:
        return None, None
    return float(np.mean(values)), (float(np.std(values, ddof=1)) if len(values) > 1 else 0.0)


PAIRED = []
for lam in LAMBDAS:
    for seed in SEEDS:
        base = RESULTS.get(f"baseline seed{seed}")
        aux = RESULTS.get(f"lambda{lambda_tag(lam)} seed{seed}")
        if base is None or aux is None:
            continue
        PAIRED.append({"lambda": lam, "seed": seed,
                       **{f"baseline_{m}": base[m] for m in METRICS},
                       **{f"aux_{m}": aux[m] for m in METRICS},
                       **{f"delta_{m}": aux[m] - base[m] for m in METRICS}})

VERDICTS = {}
for lam in LAMBDAS:
    rows = [p for p in PAIRED if p["lambda"] == lam]
    entry = {"n_seeds": len(rows)}
    for m in METRICS:
        entry[f"aux_{m}_mean"], entry[f"aux_{m}_sd"] = mean_sd([p[f"aux_{m}"] for p in rows])
        entry[f"delta_{m}_mean"], entry[f"delta_{m}_sd"] = mean_sd([p[f"delta_{m}"] for p in rows])
    if len(rows) < len(SEEDS):
        entry["verdict"] = f"INCOMPLETE ({len(rows)}/{len(SEEDS)} seeds)"
    else:
        positive = sum(1 for p in rows if p["delta_auroc_g4_vs_g3"] > 0)
        improved = (positive >= REQUIRED_POSITIVE_SEEDS
                    and entry["delta_auroc_g4_vs_g3_mean"] >= MIN_MEAN_AUROC_GAIN)
        preserved = (entry["delta_qwk_mean"] >= -MAX_MEAN_QWK_DROP
                     and entry["delta_mae_mean"] <= MAX_MEAN_MAE_RISE)
        entry.update(seeds_positive=positive, boundary_improved=improved,
                     performance_preserved=preserved,
                     grade3_recall_caution=entry["delta_grade3_recall_mean"] < GRADE3_RECALL_CAUTION,
                     verdict=("SUPPORTIVE" if improved and preserved
                              else "TRADE_OFF" if improved else "NOT_SUPPORTIVE"))
    VERDICTS[lam] = entry

BASELINE_SUMMARY = {m: mean_sd([RESULTS[f"baseline seed{s}"][m] for s in SEEDS
                                if f"baseline seed{s}" in RESULTS]) for m in METRICS}

complete_verdicts = [v for v in VERDICTS.values() if not v["verdict"].startswith("INCOMPLETE")]
n_supportive = sum(1 for v in complete_verdicts if v["verdict"] == "SUPPORTIVE")
n_improved = sum(1 for v in complete_verdicts if v.get("boundary_improved"))
if not FINAL:
    OVERALL = f"PARTIAL ({N_COMPLETE}/{len(MANIFEST)} runs complete) -- no final decision yet"
elif n_supportive >= MIN_LAMBDAS_FOR_OVERALL:
    OVERALL = "SUPPORTIVE"
elif n_improved >= MIN_LAMBDAS_FOR_OVERALL:
    OVERALL = "TRADE_OFF"
else:
    OVERALL = "NOT_SUPPORTIVE"

print(f"Completed runs: {N_COMPLETE}/{len(MANIFEST)}")
for lam, v in VERDICTS.items():
    if v["verdict"].startswith("INCOMPLETE"):
        print(f"lambda {lambda_tag(lam)}: {v['verdict']}")
    else:
        print(f"lambda {lambda_tag(lam)}: dAUROC g4-vs-g3 {v['delta_auroc_g4_vs_g3_mean']:+.4f} "
              f"({v['seeds_positive']}/3 seeds up), dQWK {v['delta_qwk_mean']:+.4f}, "
              f"dMAE {v['delta_mae_mean']:+.4f} -> {v['verdict']}")
print("OVERALL:", OVERALL)

In [ ]:
# ==== [10] FINAL REPORT GENERATION ====
# Written into the experiment root on every run of the notebook; labelled PARTIAL until 12/12.

def fmt(value, digits=4):
    return "n/a" if value is None else f"{value:.{digits}f}"


def fmt_ms(pair, digits=4):
    return "n/a" if pair is None or pair[0] is None else f"{pair[0]:.{digits}f} ± {pair[1]:.{digits}f}"


pd.DataFrame(PAIRED).to_csv(posixpath.join(EXPERIMENT_DIR, "comparison.csv"), index=False)
pd.DataFrame(list(RESULTS.values())).drop(
    columns=["val_qwk_history", "val_loss_history", "loss_history", "confusion_matrix"],
    errors="ignore").to_csv(posixpath.join(EXPERIMENT_DIR, "per_run_results.csv"), index=False)
with open(posixpath.join(EXPERIMENT_DIR, "results.json"), "w") as fh:
    json.dump({"generated": datetime.datetime.now().isoformat(timespec="seconds"),
               "final": FINAL, "completed_runs": N_COMPLETE, "overall": OVERALL,
               "preregistration": PREREGISTRATION, "per_run": RESULTS, "paired": PAIRED,
               "verdicts_by_lambda": {lambda_tag(k): v for k, v in VERDICTS.items()},
               "baseline_summary": BASELINE_SUMMARY, "environment": env_report},
              fh, indent=2, default=lambda o: float(o) if isinstance(o, np.floating)
              else (int(o) if isinstance(o, np.integer)
                    else (bool(o) if isinstance(o, np.bool_) else str(o))))

L = [f"# Grade-4 auxiliary loss — {'FINAL' if FINAL else 'PARTIAL'} report", "",
     f"Generated {datetime.datetime.now().isoformat(timespec='seconds')}. "
     f"Runs complete: **{N_COMPLETE}/{len(MANIFEST)}**.", "",
     "> The threshold pathway remains a separate calibration finding. This experiment tests whether "
     "changing the training objective improves the underlying Grade-3/Grade-4 representation.", "",
     "## Research question", "",
     "Can changing the TRAINING OBJECTIVE make the learned `p_gt_3` recover Grade-3/Grade-4 "
     "information already present in the lesion inputs? The prior diagnostic found `p_gt_3` direct "
     "AUROC ≈ 0.5979 on Grade 4 vs Grade 3, against 0.6700 for a lesion-only probe and 0.6826 for "
     "`p_gt_3` + lesion.", "",
     "## Hypothesis", "", PREREGISTRATION["hypothesis"], "",
     "## Pre-registration (frozen before any auxiliary run)", "",
     f"- lambda: **{', '.join(lambda_tag(l) for l in LAMBDAS)}**; seeds: "
     f"**{', '.join(str(s) for s in SEEDS)}**",
     f"- Loss: {PREREGISTRATION['loss_auxiliary']}",
     f"- Baseline: {PREREGISTRATION['baseline_policy']}",
     f"- Checkpoint selection: {PREREGISTRATION['checkpoint_selection']}",
     f"- Primary outcome: {PREREGISTRATION['primary_outcome']}",
     f"- Decision: boundary improved = {PREREGISTRATION['decision_rule']['boundary_improved']}; "
     f"performance preserved = {PREREGISTRATION['decision_rule']['performance_preserved']}; "
     f"{PREREGISTRATION['decision_rule']['per_lambda']}. Overall: "
     f"{PREREGISTRATION['decision_rule']['overall']}",
     "- Everything else (Stage 01–07, segmentation, RACAF, reliability, preprocessing, split, "
     "augmentation, optimizer, LR schedule, regularisation, backbone, mixed precision) unchanged.",
     "", "## Run manifest and status", "", "| # | run | status | source |", "|---|---|---|---|"]
for row in write_status_table():
    L.append(f"| {row['index']} | {row['label']} | {row['status']} | `{row['result_dir']}` |")

L += ["", "## Per-run results (BEST checkpoint)", "",
      "| run | g4-vs-g3 AUROC | g4-vs-rest AUROC | g4 recall | g4 prec. | g4 F1 | g3 recall | QWK | "
      "acc | bal. acc | macro F1 | MAE | best ep. | epochs |",
      "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
for run in MANIFEST:
    r = RESULTS.get(run["label"])
    if r is None:
        L.append(f"| {run['label']} | " + " | ".join(["—"] * 13) + " |")
        continue
    L.append(f"| {run['label']} | {fmt(r['auroc_g4_vs_g3'])} | {fmt(r['auroc_g4_vs_rest'])} | "
             f"{fmt(r['grade4_recall'])} | {fmt(r['grade4_precision'])} | {fmt(r['grade4_f1'])} | "
             f"{fmt(r['grade3_recall'])} | {fmt(r['qwk'])} | {fmt(r['accuracy'])} | "
             f"{fmt(r['balanced_accuracy'])} | {fmt(r['macro_f1'])} | {fmt(r['mae'])} | "
             f"{r.get('best_epoch_index_0based')} | {r.get('epochs_trained')} |")

L += ["", "## Baseline (mean ± SD across seeds)", "",
      ", ".join(f"{m} {fmt_ms(BASELINE_SUMMARY[m])}" for m in
                ("auroc_g4_vs_g3", "auroc_g4_vs_rest", "grade4_recall", "grade3_recall", "qwk",
                 "mae", "macro_f1")), "",
      "## Grade 3 vs Grade 4 — the primary comparison (paired by seed)", "",
      "| lambda | seed | baseline | auxiliary | Δ |", "|---|---|---|---|---|"]
for p in PAIRED:
    L.append(f"| {lambda_tag(p['lambda'])} | {p['seed']} | {fmt(p['baseline_auroc_g4_vs_g3'])} | "
             f"{fmt(p['aux_auroc_g4_vs_g3'])} | {p['delta_auroc_g4_vs_g3']:+.4f} |")

L += ["", "## Per-lambda summary (auxiliary mean ± SD; Δ vs matched baseline)", "",
      "| lambda | g4-vs-g3 AUROC Δ | g4-vs-rest AUROC Δ | g4 recall Δ | g3 recall Δ | QWK Δ | MAE Δ | "
      "macro F1 Δ | verdict |", "|---|---|---|---|---|---|---|---|---|"]
for lam, v in VERDICTS.items():
    if v["verdict"].startswith("INCOMPLETE"):
        L.append(f"| {lambda_tag(lam)} | " + " | ".join(["—"] * 7) + f" | {v['verdict']} |")
        continue
    caution = " ⚠ grade-3 recall" if v.get("grade3_recall_caution") else ""
    L.append(f"| {lambda_tag(lam)} | {fmt_ms((v['delta_auroc_g4_vs_g3_mean'], v['delta_auroc_g4_vs_g3_sd']))} | "
             f"{fmt_ms((v['delta_auroc_g4_vs_rest_mean'], v['delta_auroc_g4_vs_rest_sd']))} | "
             f"{fmt_ms((v['delta_grade4_recall_mean'], v['delta_grade4_recall_sd']))} | "
             f"{fmt_ms((v['delta_grade3_recall_mean'], v['delta_grade3_recall_sd']))} | "
             f"{fmt_ms((v['delta_qwk_mean'], v['delta_qwk_sd']))} | "
             f"{fmt_ms((v['delta_mae_mean'], v['delta_mae_sd']))} | "
             f"{fmt_ms((v['delta_macro_f1_mean'], v['delta_macro_f1_sd']))} | "
             f"**{v['verdict']}**{caution} |")

L += ["", "## Grade 4 vs rest, QWK and MAE", "",
      "Grade-4-vs-rest AUROC, Grade-4 recall, QWK and MAE are reported above per run and per "
      "lambda. They are secondary: a lambda is never judged successful because Grade-4 recall rose, "
      "and QWK is known to be nearly blind to 3↔4 swaps (a distance-1 error carries 1/16 the weight "
      "of a distance-4 error), which is why it serves only as a guard against broad degradation.",
      "", "## Auxiliary-loss contribution", "",
      "`val_grade4_bce_at_best` is the auxiliary BCE on the validation set at BEST, computed from "
      "`p_gt_3` identically for baseline and auxiliary runs (the baseline never optimised it). The "
      "per-epoch logged value is in each run's `history_full/`. `val_loss` includes λ·BCE for "
      "auxiliary runs, so `val_corn_loss_unweighted` is the like-for-like CORN comparison.", "",
      "| run | val BCE at BEST | val CORN loss (unweighted) at BEST |", "|---|---|---|"]
for run in MANIFEST:
    r = RESULTS.get(run["label"])
    if r is not None:
        L.append(f"| {run['label']} | {fmt(r['val_grade4_bce_at_best'])} | "
                 f"{fmt(r.get('val_corn_loss_unweighted_at_best'))} |")

L += ["", "## Limitations", "",
      "- 39 Grade-3 and 58 Grade-4 validation cases: a single-seed AUROC on that contrast has an SE "
      "near 0.058, which is why 3/3 sign consistency is required.",
      "- The auxiliary BCE is unweighted against a ~8% positive class, so this is a deliberately mild "
      "intervention; a null result is partly attributable to that design choice.",
      "- BEST is selected on val_QWK exactly as for the baselines. That keeps the comparison fair "
      "but may disfavour an objective aimed at a boundary QWK barely weighs.",
      "- Three lambdas are tested: a single passing lambda is weaker evidence than it looks, which "
      f"is why the overall verdict needs {MIN_LAMBDAS_FOR_OVERALL}. No lambda is selected "
      "retrospectively.",
      "- GPU training is not bit-deterministic, so matched seeds match initialisation and data order, "
      "not the full trajectory.",
      "- One frozen APTOS split; no external validation.", "",
      f"## Diagnostic decision: **{OVERALL}**", ""]
if FINAL:
    L.append("Per lambda: " + "; ".join(f"{lambda_tag(k)} → {v['verdict']}"
                                        for k, v in VERDICTS.items()) + ".")
    single = [lambda_tag(k) for k, v in VERDICTS.items() if v["verdict"] == "SUPPORTIVE"]
    if len(single) == 1:
        L.append("")
        L.append(f"Only lambda {single[0]} met the per-lambda criterion. Under the pre-registered "
                 "rule a single lambda out of three is not sufficient for an overall SUPPORTIVE "
                 "verdict; it is reported, not selected.")
else:
    L.append("The decision is withheld until all 12 runs are complete.")

REPORT_PATH = posixpath.join(EXPERIMENT_DIR, "REPORT.md")
with open(REPORT_PATH, "w") as fh:
    fh.write("\n".join(L) + "\n")
print("Wrote", REPORT_PATH)
print("Also:", posixpath.join(EXPERIMENT_DIR, "comparison.csv"), "|",
      posixpath.join(EXPERIMENT_DIR, "per_run_results.csv"), "|",
      posixpath.join(EXPERIMENT_DIR, "results.json"))